In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import *
from datetime import date, timedelta, datetime
from pyspark.sql.types import *
import random

In [0]:


spark = SparkSession.builder.getOrCreate()

customers = [
    (i, f"Customer_{i}", random.choice(["India", "USA", "Germany", "UK", "Canada"]))
    for i in range(1, 1001)
]

customer_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("country", StringType(), True)
])

df_customers = spark.createDataFrame(customers, customer_schema)
df_customers.show(5)


In [0]:


orders = [
    (
        i,
        random.randint(1, 1000),
        date(2024, 1, 1) + timedelta(days=random.randint(0, 90)),
        round(random.uniform(100, 5000), 2)
    )
    for i in range(1, 50001)
]

order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", DateType(), True),
    StructField("order_amount", DoubleType(), True)
])

df_orders = spark.createDataFrame(orders, order_schema)
df_orders.show(5)


In [0]:
payments = []

for i in range(1, 70001):
    payments.append((
        random.randint(1, 50000),
        random.choice(["SUCCESS", "FAILED"]),
        round(random.uniform(50, 5000), 2),
        datetime.now()
    ))

payment_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("payment_status", StringType(), True),
    StructField("payment_amount", DoubleType(), True),
    StructField("payment_time", TimestampType(), True)
])

df_payments = spark.createDataFrame(payments, payment_schema)
df_payments.show(5)


In [0]:
display(df_customers)
display(df_orders)
display(df_payments)

Q1 – Join Fundamentals

Join customers and orders

Output:

customer_name

country

order_id

order_date

order_amount

In [0]:
q1 = df_orders.join(broadcast(df_customers), df_orders.customer_id == df_customers.customer_id, "left") \
            .select(df_customers.customer_name, df_customers.country, df_orders.order_id, df_orders.order_date, df_orders.order_amount)

display(q1)

Move to Q2 – Aggregation at Scale
Q2 Reminder

Find total order amount per customer

Show top 10 customers globally

In [0]:
q2 = df_orders.alias("o").join(df_customers.alias("c"), col("c.customer_id") == col("o.customer_id"), "left") \
            .groupBy(col("c.customer_name")) \
            .agg(sum(col("o.order_amount")).alias("total_order"))\
            .orderBy(col("total_order").desc()) \
            .limit(10)

display(q2)

Q3 – Window Function (No Aggregation)

For each customer, find the highest-value order.

Rules:

Must use a window function

No groupBy

Justify row_number vs rank

In [0]:
window = (
    Window
    .partitionBy("o.customer_id")
    .orderBy(F.col("o.order_amount").desc())
)

q3 = (
    df_orders.alias("o")
    .join(
        df_customers.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .select(
        "c.customer_id",
        "c.customer_name",
        "o.order_id",
        "o.order_amount"
    )
)

display(q3)

Q4 – Dedup Payments

Payments table has:

Multiple rows per order

SUCCESS + FAILED

Multiple SUCCESS attempts

Task

Keep only latest SUCCESS payment per order

Use window function

Ignore FAILED payments

In [0]:
# Get the dataframe which only has one success payment
# combine with orders table and get the payment amount.

window = Window.partitionBy("p.order_id").orderBy(col("p.payment_time").desc())

pfinal = df_payments.alias("p").filter(col("payment_status") == "SUCCESS") \
            .withColumn("rank", row_number().over(window)) \
            .filter(col("rank") == 1 )

q4 = df_orders.alias("o").join(pfinal.alias("p"), col("o.order_id") == col("p.order_id"), "left") \
        .select(col("o.order_id"), col("p.payment_status"), col("p.payment_amount"))


display(q4)



In [0]:
o_test = df_orders.filter(col("order_id") == 11451)
display(o_test)
p_test = df_payments.groupBy("order_id") \
            .agg(count("order_id").alias("count"))

p_test1 = df_payments.filter(col("order_id") == 11451)
display(p_test1)